In [1]:
import tensorflow as tf
from tensorflow import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.18.1
Keras: 3.15.1
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
MAX_UNKNOWN = 1.0 # 1.0 = keep every training tile try 0.5 to drop tiles

In [3]:
import sys
sys.path.append("../scripts")
from dataset import get_tiles, SPLITS
X_train, y_train = get_tiles(SPLITS["train"], stride=128,
                             max_unknown=MAX_UNKNOWN)

# Val and test no overlap, keep every tile (the defaults)
X_val, y_val = get_tiles(SPLITS["val"])
X_test, y_test = get_tiles(SPLITS["test"])

In [4]:
import numpy as np

for name, X, y in [("Train", X_train, y_train), ("Val", X_val, y_val), ("Test", X_test, y_test)]:
    unknown = (y == -1).mean()
    known = y[y != -1]
    print(f"{name:5}  X {X.shape}  y {y.shape}  unknown {unknown:.1%}  flood (of known) {known.mean():.1%}")

Train  X (1972, 256, 256, 4)  y (1972, 256, 256)  unknown 23.7%  flood (of known) 55.7%
Val    X (110, 256, 256, 4)  y (110, 256, 256)  unknown 20.0%  flood (of known) 51.3%
Test   X (287, 256, 256, 4)  y (287, 256, 256)  unknown 48.5%  flood (of known) 76.0%


In [5]:
#make copy of the training tiles that keeps only the rgb channels and leaves out the dem part

X_train_rgb = X_train[..., :3]
X_val_rgb = X_val[..., :3]
X_test_rgb = X_test[..., :3]

print(X_train_rgb.shape, X_val_rgb.shape, X_test_rgb.shape)
print("RGB min:", X_train_rgb.min(), "max:", X_train_rgb.max())

(1972, 256, 256, 3) (110, 256, 256, 3) (287, 256, 256, 3)
RGB min: 0.0 max: 1.0


## Prepare the images for ResNet50

ResNet50 was pretrained on images in the 0-255 range, preprocessed with `preprocess_input`
(RGB flipped to BGR, ImageNet mean color subtracted). Our RGB is 0-1, so multiply by 255 first.

In [6]:
from tensorflow.keras.applications.resnet50 import preprocess_input

def preprocess(x_rgb):
    return preprocess_input(x_rgb * 255.0) #0-1 to 0-255

X_train_in = preprocess(X_train_rgb)
X_val_in = preprocess(X_val_rgb)
X_test_in = preprocess(X_test_rgb)

print(X_train_in.shape)
print("min:", X_train_in.min(), "max:", X_train_in.max())

(1972, 256, 256, 3)
min: -123.68 max: 151.061


## Prepare the Labels

In [7]:
def prepare_labels(y):
    return y.astype("float32")[..., None] # Changes (N, 256, 256) to (N, 256, 256, 1)

y_train_in = prepare_labels(y_train)
y_val_in = prepare_labels(y_val)
y_test_in = prepare_labels(y_test)

print(y_train_in.shape)
print("values:", np.unique(y_train_in))

(1972, 256, 256, 1)
values: [-1.  0.  1.]


## Loss function (skips unknown pixels)

In [8]:
def masked_bce(y_true, y_pred):
    known = tf.cast(y_true != -1, "float32") # 1 = labeled pixel, 0 = unknown, turns T/F into 1.0/0.0
    y_true = tf.where(y_true == -1, 0.0, y_true) # -1 would break the loss , so set iti to 0
                                                 # replaces every -1 with 0. BCE only understands labels of 0 or 1, and a -1 would give nonsense.
    loss = keras.losses.binary_crossentropy(y_true, y_pred)[..., None] #computes loss for every pixel
 
    return tf.reduce_sum(loss * known) / tf.reduce_sum(known) #average over labeled picels only
                                                              #loss * known zeros out the pentalty on unknown pixels
                                                              #tf.reduce_sum / tf.reduce_sum adds up the remaining penalties and divides by how many pixels are known.
                                                              #gives the average penallty over labeled pixels only



In [10]:
y_true = tf.constant([[[[-1.0], [1.0]], [[0.0], [1.0]]]])   # unknown, flood, dry, flood
y_pred = tf.constant([[[[0.9], [0.9]], [[0.1], [0.8]]]])    # model's guesses
print(masked_bce(y_true, y_pred).numpy())

0.14462154


## Build the model: ResNet50 encoder
## Tutorial Repo this code came from:
https://github.com/nikhilroxtomar/Semantic-Segmentation-Architecture/blob/main/TensorFlow/resnet50_unet.py

In [ ]:
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, Conv2DTranspose, Concatenate, Input
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50

def conv_block(x, num_filters):
    x = Conv2D(num_filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Conv2D(num_filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x

def decoder_block(x, skip_features, num_filters):
    x = Conv2DTranspose(num_filters, 2, strides=2, padding="same")(x)   # learned 2x upsample
    x = Concatenate()([x, skip_features])
    return conv_block(x, num_filters)

def build_resnet50_unet(input_shape):
    inputs = Input(input_shape)
    resnet50 = ResNet50(include_top=False, weights="imagenet", input_tensor=inputs)

    # Encoder skips (sizes for 256 x 256 tiles)
    s1 = inputs                                          # 256 
    s2 = resnet50.get_layer("conv1_relu").output         # 128
    s3 = resnet50.get_layer("conv2_block3_out").output   # 64
    s4 = resnet50.get_layer("conv3_block4_out").output   # 32

    # Bridge
    b1 = resnet50.get_layer("conv4_block6_out").output   # 16

    # Decoder
    d1 = decoder_block(b1, s4, 512)   # 32
    d2 = decoder_block(d1, s3, 256)   # 64
    d3 = decoder_block(d2, s2, 128)   # 128
    d4 = decoder_block(d3, s1, 64)    # 256

    outputs = Conv2D(1, 1, padding="same", activation="sigmoid")(d4)
    return Model(inputs, outputs, name="ResNet50_U-Net"), resnet50

model, encoder = build_resnet50_unet((256, 256, 3))
print(model.input_shape, "->", model.output_shape)

(None, 256, 256, 3) -> (None, 256, 256, 1)
